# Cryo-EM benchmark

In [ ]:
import csv
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from ctypes.util import find_library
from importlib.util import find_spec
from pathlib import Path

ROOT = Path.cwd()
WORK = Path("/workspace/cryo-benchmark-work")
PHENIX_INSTALLER = ROOT / "Phenix-2.2.1-6174-Linux-x86_64.sh"
PHENIX_ENV = WORK / "phenix" / "phenix_env.sh"
SOURCES = WORK / "sources"
ENVS = WORK / "envs"
RUNS = WORK / "runs"
METHODS = ("MICA", "ModelAngelo", "EModelX", "CryoAtom", "EMProt")
CPU_WORKERS = 4

targets = {}
for name, emdb_id, resolution in [("22XP", "68757", 3.20), ("32MC", "58994", 2.95)]:
    folder = ROOT / "data" / "targets" / name
    job = f"fold_{name.lower()}_1"
    target = {
        "map": folder / "inputs" / f"emd_{emdb_id}.map",
        "fasta": folder / "inputs" / f"rcsb_pdb_{name}.fasta",
        "af2": next(folder.glob("predictions/af2/*/*_unrelaxed_rank_001_*.pdb")),
        "af3": folder / "predictions" / "af3" / job / f"{job}_model_0.cif",
        "reference": folder / "reference" / f"{name}.cif",
    }
    for file in target.values():
        if not file.is_file():
            raise FileNotFoundError(file)
    targets[name] = dict(target, emdb_id=emdb_id, contour=0.06, resolution=resolution)

In [ ]:
os.environ.update(
    MPLBACKEND="Agg",
    TORCH_HOME=str(WORK / "torch-cache"),
    CONDA_CHANNEL_PRIORITY="flexible",
    CONDA_PKGS_DIRS=str(WORK / "conda-pkgs"),
    UV_CACHE_DIR=str(WORK / "uv-cache"),
)
os.environ.pop("UV_SYSTEM_PYTHON", None)


def run(args, log=None, **options):
    if log is not None:
        options.update(stdout=log, stderr=subprocess.STDOUT)
    return subprocess.run([str(arg) for arg in args], check=True, text=True, **options)


def checkout(name, url, revision):
    repo = SOURCES / name
    if repo.exists():
        run(["git", "-C", repo, "checkout", "--quiet", "--detach", revision])
    else:
        with tempfile.TemporaryDirectory(dir=SOURCES) as staging:
            clone = Path(staging) / name
            run(["git", "clone", "--no-checkout", url, clone])
            run(["git", "-C", clone, "checkout", "--quiet", "--detach", revision])
            clone.rename(repo)
    return repo


def unpack(url, filename, destination):
    archive = WORK / filename
    run(["curl", "-fL", "--retry", "3", url, "-o", archive])
    shutil.unpack_archive(archive, destination)
    archive.unlink()


def uv_command():
    if shutil.which("uv"):
        return ["uv", "--no-config"]
    if find_spec("uv") is None:
        run([sys.executable, "-m", "pip", "install", "uv"])
    return [sys.executable, "-m", "uv", "--no-config"]


def uv_environment(name, cuda, torch, packages):
    prefix = ENVS / name
    python = prefix / "bin" / "python"
    if not (prefix / ".ready").exists():
        uv = uv_command()
        if not python.exists():
            run([*uv, "venv", "--python", "3.11", prefix])
        pip = [*uv, "pip", "install", "--python", python]
        run([*pip, "--index-url", f"https://download.pytorch.org/whl/{cuda}", *torch])
        run([*pip, *packages])
        (prefix / ".ready").touch()
    return prefix / "bin"


def conda_environment(repo, file):
    conda = Path(shutil.which("conda") or WORK / "miniforge" / "bin" / "conda")
    if not conda.exists():
        installer = WORK / "miniforge.sh"
        run(["curl", "-fL", "--retry", "3",
             "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh",
             "-o", installer])
        run(["bash", installer, "-b", "-p", conda.parent.parent])
        installer.unlink()

    prefix = ENVS / repo.name
    ready = prefix / ".ready"
    if not ready.exists():
        action = "update" if (prefix / "conda-meta" / "history").exists() else "create"
        run([conda, "env", action, "-p", prefix, "-f", repo / file])
        ready.touch()
    return [conda, "run", "-p", prefix, "--no-capture-output"]


def install_phenix():
    if not PHENIX_ENV.is_file():
        run(["bash", PHENIX_INSTALLER, "-b", "-p", PHENIX_ENV.parent])


def phenix(*args):
    return ["bash", "-c", 'source "$1" && shift && exec "$@"', "phenix", PHENIX_ENV, *args]


def benchmark(method, build):
    for name, target in targets.items():
        folder = RUNS / name / method
        folder.mkdir(parents=True, exist_ok=True)
        record = folder / "result.json"
        if record.exists():
            model = json.loads(record.read_text()).get("model")
            if model and Path(model).is_file():
                continue

        attempt = Path(tempfile.mkdtemp(prefix="run-", dir=folder))
        log_path = attempt / "run.log"
        print(f"{name} {method} → {log_path}", flush=True)
        start = time.perf_counter()
        result = {"log": str(log_path)}
        try:
            with log_path.open("w") as log:
                model = build(target, attempt, log)
            if not model.is_file() or model.stat().st_size == 0:
                raise FileNotFoundError(model)
            result.update(status="ok", model=str(model))
        except (OSError, subprocess.CalledProcessError) as error:
            result.update(status="failed", error=str(error))
            print(error)
        result["runtime_seconds"] = round(time.perf_counter() - start, 1)
        record.write_text(json.dumps(result))

In [ ]:
for folder in (SOURCES, ENVS, RUNS):
    folder.mkdir(parents=True, exist_ok=True)

packages = [name for name in ("git", "curl", "make", "g++") if not shutil.which(name)]
if not find_library("GL"):
    packages.append("libgl1")
if not find_library("gfortran"):
    packages.append("libgfortran5")
if packages:
    run(["apt-get", "update"])
    run(["apt-get", "install", "-y", *packages])

In [ ]:
# MICA (README Option B)
install_phenix()
mica = checkout("MICA", "https://github.com/jianlin-cheng/MICA.git", "726d1bb")
mica_bin = uv_environment("MICA", "cu124", ["torch==2.5.1"], [
    "numpy==1.26.4", "scipy==1.13.1", "biopython==1.83", "open3d-cpu==0.19.0",
    "superpose3d==1.1.1", "networkx==3.4.2", "rotary-embedding-torch==0.8.9",
    "mrcfile", "psutil", "tqdm", "einops", "natsort", "matplotlib", "requests",
])
if not (mica / "trained_models" / "MICA_best_model.pth").is_file():
    unpack("https://zenodo.org/records/15756654/files/trained_models.tar.gz?download=1",
           "mica.tar.gz", mica)


def build_mica(target, attempt, log):
    af3 = attempt / "AF3_results"
    job = af3 / target["af3"].parent.name
    job.mkdir(parents=True)
    shutil.copy2(target["af3"], job)
    out = attempt / "model"
    map_file, fasta, resolution = (target[key] for key in ("map", "fasta", "resolution"))
    python = [mica_bin / "python", "-u"]
    mica_env = dict(os.environ, PATH=f"{mica_bin}{os.pathsep}{os.environ['PATH']}")
    options = dict(cwd=mica, log=log, env=mica_env)

    run([*python, "utils/process_AF3_results.py", "-f", fasta, "-a", af3, "-d", "cuda:0"], **options)
    run([*python, "utils/dock_in_map.py", "-m", map_file, "-f", fasta, "-a", af3,
         "-c", target["contour"], "-r", resolution, "-x", PHENIX_ENV], **options)
    docked = attempt / f"{attempt.name}_af3_docked.pdb"
    if not docked.is_file():
        raise FileNotFoundError(docked)
    run([*python, "run.py", "-m", map_file, "-f", fasta, "-a", af3, "-r", resolution, "-o", out,
         "--device", "cuda:0", "--run_pulchra", "-p", "modules/pulchra304/src/pulchra",
         "--no_parallel", "--mul_proc_num", CPU_WORKERS], **options)

    raw = out / f"{target['emdb_id']}_{fasta.stem}_MICA_all_atom_model.pdb"
    run(phenix("phenix.real_space_refine", raw, map_file, mica / "modules" / "phenix.eff",
               f"resolution={resolution}", f"nproc={CPU_WORKERS}"), cwd=out, log=log)
    refined = sorted(out.glob(f"{raw.stem}_real_space_refined*"))
    for file in refined:
        if file.suffix in (".pdb", ".cif"):
            return file
    raise FileNotFoundError(f"Missing refined model in {out}")


benchmark("MICA", build_mica)

In [ ]:
# ModelAngelo
angelo = checkout("ModelAngelo", "https://github.com/3dem/model-angelo.git", "994945b")
angelo_bin = uv_environment("ModelAngelo", "cu126", ["torch==2.9.1", "torchvision==0.24.1"], [angelo])
angelo_cli = angelo_bin / "model_angelo"
run([angelo_cli, "setup_weights", "--bundle-name", "nucleotides"])


def build_angelo(target, attempt, log):
    out = attempt / "model"
    run([angelo_cli, "build", "-v", target["map"], "-pf", target["fasta"], "-o", out, "-d", "0"],
        cwd=angelo, log=log)
    return out / "model.cif"


benchmark("ModelAngelo", build_angelo)

In [ ]:
# EModelX(+AF)
emodelx = checkout("EModelX", "https://github.com/biomed-AI/EModelX.git", "53713cf")
emodelx_run = conda_environment(emodelx, "EModelX.yml")
weights = emodelx / "models"
weights.mkdir(exist_ok=True)
for atom, file_id in [
    ("CA", "1n8yiVr4oRxsY4ii2a05hcx5gI3eIybA1"),
    ("BB", "1GbmezR4tVCYaALZ330TL4vk84kGDQD3V"),
    ("AA", "1evghUiUcb3mFiCQ2-Gjez9AtA2w6Pyir"),
]:
    file = weights / f"best_{atom}_model.ckpt"
    if not file.is_file():
        partial = file.with_suffix(".part")
        run([*uv_command(), "tool", "run", "gdown", file_id, "-O", partial])
        partial.replace(file)


def build_emodelx(target, attempt, log):
    templates = attempt / "templates"
    fasta = target["fasta"]
    fasta_id = fasta.read_text().splitlines()[0][1:].split("|")[0]
    template = templates / fasta_id / "ranked_0.pdb"
    template.parent.mkdir(parents=True)
    shutil.copy2(target["af2"], template)
    out = attempt / "model"
    run([*emodelx_run, "python", "-u", "run.py", "--protocol", "temp_flex",
         "--EM_map", target["map"], "--fasta", fasta, "--template_dir", templates,
         "--output_dir", out, "--run_pulchra", "--pulchra_path", "modules/pulchra304/src/pulchra",
         "--mul_proc_num", CPU_WORKERS], cwd=emodelx, log=log)
    return out / f"{fasta.stem}_EModelX(+AF)_all_atom_model.pdb"


benchmark("EModelX", build_emodelx)

In [ ]:
# CryoAtom v1.0
cryoatom = checkout("CryoAtom", "https://github.com/YangLab-SDU/CryoAtom.git", "v1.0.0")
cryoatom_run = conda_environment(cryoatom, "linux.yml")
checkpoints = cryoatom / "CryoAtom" / "checkpoint"
if not all((checkpoints / name).is_file() for name in ("CryNet.pth", "SimpleUnet.pth")):
    unpack("https://yanglab.qd.sdu.edu.cn/CryoAtom/download/checkpoints_v1.0.zip",
           "cryoatom.zip", cryoatom)
if not (ENVS / "CryoAtom" / "bin" / "cryoatom").is_file():
    run([*cryoatom_run, "python", "-m", "pip", "install", "."], cwd=cryoatom)


def build_cryoatom(target, attempt, log):
    out = attempt / "model"
    run([*cryoatom_run, "cryoatom", "build", "-v", target["map"], "-s", target["fasta"],
         "-o", out, "-d", "cuda:0"], cwd=cryoatom, log=log)
    return out / "model.cif"


benchmark("CryoAtom", build_cryoatom)

In [ ]:
# EMProt
emprot = checkout("EMProt", "https://github.com/huang-laboratory/EMProt.git", "9489d52")
emprot_run = conda_environment(emprot, "environment.yml")
if not (ENVS / "EMProt" / "bin" / "emprot").is_file():
    run([*emprot_run, "python", "-m", "pip", "install", "-e", "."], cwd=emprot)
if not (emprot / "emprot" / "weights" / "model_all_atom").is_dir():
    unpack("http://huanglab.phys.hust.edu.cn/EMProt/pretrained_weights/weights.tgz",
           "emprot.tar.gz", emprot / "emprot")

# EMProt takes a single chain per template file.
extract_chain = '''
import sys
from Bio.PDB import MMCIFParser, PDBIO
structure = MMCIFParser(QUIET=True).get_structure("af3", sys.argv[1])
writer = PDBIO()
writer.set_structure(next(structure.get_chains()))
writer.save(sys.argv[2])
'''


def build_emprot(target, attempt, log):
    template = attempt / "af3_chain.pdb"
    out = attempt / "model"
    run([*emprot_run, "python", "-c", extract_chain, target["af3"], template], cwd=emprot, log=log)
    run([*emprot_run, "emprot", "build", "--map", target["map"], "--seq", target["fasta"],
         "--chain", template, "--device", "0", "--output", out], cwd=emprot, log=log)
    return out / "output.cif"


benchmark("EMProt", build_emprot)

In [ ]:
# Score against the deposited structures (full assemblies).
install_phenix()
usalign = checkout("USalign", "https://github.com/pylelab/USalign.git", "23f14ff")
run(["make", "-C", usalign, "USalign"])

chain_comparison = '''
import json
import sys
from libtbx.utils import null_out
from mmtbx.validation.chain_comparison import run
scores = run(args=[sys.argv[1], sys.argv[2], "comparison.max_dist=3.0",
                   "input_files.unique_part_of_target_only=False",
                   "input_files.test_unique_part_of_target_only=False",
                   "crystal_info.use_crystal_symmetry=False"], out=null_out())
matched = scores.get_values("close")[1]
print(json.dumps({"ca_match_pct": 100 * matched / scores.used_chain,
                  "ca_quality_pct": 100 * matched / scores.used_target,
                  "sequence_match_pct": scores.get_match_percent("close") if matched else 0.0}))
'''


def score(model, reference):
    text = run([usalign / "USalign", model, reference, "-mol", "prot", "-mm", "1",
                "-ter", "1", "-outfmt", "2"], capture_output=True).stdout
    aligned = next(csv.DictReader(text.splitlines(), delimiter="\t"))
    comparison = run(phenix("phenix.python", "-c", chain_comparison, reference, model), capture_output=True)
    return dict(tm_score=float(aligned["TM2"]), aligned_ca=int(aligned["Lali"]),
                sequence_identity=float(aligned["IDali"]),
                **json.loads(comparison.stdout.splitlines()[-1]))


rows = []
for name, target in targets.items():
    for method in METHODS:
        record = RUNS / name / method / "result.json"
        row = dict(target=name, method=method, status="missing")
        if record.is_file():
            row.update(json.loads(record.read_text()))
        if row["status"] == "ok":
            try:
                row.update(score(row["model"], target["reference"]))
            except (OSError, ValueError, KeyError, IndexError, StopIteration, subprocess.CalledProcessError) as error:
                row.update(status="scoring failed", error=getattr(error, "stderr", None) or str(error))
        rows.append(row)

fields = ("target", "method", "status", "tm_score", "aligned_ca", "sequence_identity",
          "ca_match_pct", "ca_quality_pct", "sequence_match_pct", "runtime_seconds", "model", "log", "error")
csv_path = RUNS / "benchmark.csv"
with csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)
print(csv_path)